# Huấn luyện Mô hình PatchCore Phát hiện Bất thường Bo mạch (Dev 2)

Notebook này thực hiện quy trình huấn luyện (**Training**), hiệu chỉnh ngưỡng quyết định (**Threshold $\tau$ Calibration**) và xuất trọng số mô hình **PatchCore (`wide_resnet50_2`)** trên **Google Colab GPU**, tuân thủ đặc tả kỹ thuật tại `docs/developer_spec.md`.

### Quy trình thực hiện:
1. Kiểm tra GPU & Cài đặt thư viện (`torch`, `torchvision`, `anomalib`, `opencv-python-headless`, `scipy`, `pyyaml`).
2. Kết nối Google Drive / Tải bộ dữ liệu (`data/chip_dataset/`).
3. Kiểm tra & Tiền xử lý ảnh bo mạch qua `BoardAligner` (Tách nền LAB/Otsu, xoay thẳng `minAreaRect`, chống ngược 180°, cân bằng sáng CLAHE).
4. Huấn luyện **PatchCore (`wide_resnet50_2`, layers `layer2` + `layer3`)** kết hợp biến đổi **Photometric Multi-Lighting Augmentation** (không lật ảnh Horizontal/Vertical Flip).
5. Đánh giá trên tập Validation/Test (`AUROC`, `F1-Score`), tìm ngưỡng tối ưu $\tau$ và trực quan hóa **Anomaly Heatmap**.
6. Xuất và tải về bộ đôi `weights/patchcore_best.ckpt` + `weights/metadata.json` để chạy ứng dụng Streamlit trên Laptop.

In [ ]:
# 1. Kiểm tra GPU và cài đặt thư viện cần thiết trên Google Colab
!nvidia-smi
!pip install -q torch torchvision opencv-python-headless scipy pyyaml matplotlib scikit-learn anomalib

In [ ]:
# 2. Giải nén gói dự án (colab_bundle.zip) và thiết lập thư mục làm việc trên Google Colab
import os
import sys
import zipfile
from pathlib import Path

if Path("/content").exists():
    PROJECT_ROOT = Path("/content/Chip_detection")
    bundle_path = Path("/content/colab_bundle.zip")
    if not (PROJECT_ROOT / "src").exists():
        if not bundle_path.exists():
            from google.colab import files
            print("Hãy chọn tải file 'colab_bundle.zip' từ máy tính của bạn lên:")
            uploaded = files.upload()
            if uploaded:
                first_file = next(iter(uploaded.keys()))
                bundle_path = Path("/content") / first_file
        if bundle_path.exists():
            PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(bundle_path, "r") as zf:
                zf.extractall(PROJECT_ROOT)
            print(f"Đã giải nén {bundle_path} vào {PROJECT_ROOT}")
    os.chdir(PROJECT_ROOT)
elif (Path.cwd().name == "notebooks") and (Path.cwd().parent / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Working directory: {PROJECT_ROOT}")

## 3. Kiểm tra quy chuẩn Dataset & Tiền xử lý (`BoardAligner`)

Đảm bảo cấu trúc thư mục tuân thủ:
- `data/chip_dataset/normal/train/`: 30 – 50 ảnh Normal (góc 90° Top-down, 4 điều kiện hướng sáng).
- `data/chip_dataset/normal/val/`: 10 – 15 ảnh Normal độc lập.
- `data/chip_dataset/abnormal/test/`: 10 – 15 ảnh Anomaly (thiếu linh kiện SMD cạnh biến trở `103`, lệch linh kiện, v.v.).

In [ ]:
# 3. Chạy kiểm tra Dataset (hoặc tự động khởi tạo từ sample2.jpg Normal & sample1.jpg Anomaly nếu chạy thử nghiệm)
from src.training.train_patchcore import bootstrap_dataset_from_samples

counts = bootstrap_dataset_from_samples(dataset_root=Path("data/chip_dataset"))
print("Số lượng ảnh trong Dataset:", counts)

In [ ]:
# 4. Huấn luyện mô hình PatchCore (Backbone: wide_resnet50_2, Layers: layer2 + layer3, Coreset ratio: 0.1)
from src.training.train_patchcore import train_patchcore_model

metadata = train_patchcore_model(
    config_path=Path("configs/patchcore_chip.yaml"),
    dataset_root=Path("data/chip_dataset"),
    output_weights_path=Path("weights/patchcore_best.ckpt"),
    output_metadata_path=Path("weights/metadata.json"),
    auto_bootstrap=True,
)
print("Kết quả huấn luyện & hiệu chỉnh ngưỡng:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

In [ ]:
# 5. Đánh giá độc lập trên tập Validation / Test và kiểm tra trên 2 ảnh mẫu sample2.jpg (Normal) & sample1.jpg (Anomaly)
import cv2
from src.inference.inspector import InspectionModel
from src.training.evaluate import evaluate_dataset

eval_report = evaluate_dataset(
    config_path=Path("configs/patchcore_chip.yaml"),
    weights_path=Path("weights/patchcore_best.ckpt"),
    dataset_root=Path("data/chip_dataset"),
    metadata_path=Path("weights/metadata.json"),
)
print("AUROC:", eval_report["auroc"])
print("Optimal Threshold (tau):", eval_report["optimal_threshold"])
print("F1-Score:", eval_report["f1_score"])
print("Confusion Matrix:", eval_report["confusion_matrix"])

inspector = InspectionModel(
    weights_path=Path("weights/patchcore_best.ckpt"),
    metadata_path=Path("weights/metadata.json"),
)
inspector.load()

for label_name, sample_path in [("Normal (sample2.jpg)", "image/sample2.jpg"), ("Anomaly (sample1.jpg)", "image/sample1.jpg")]:
    bgr = cv2.imread(sample_path)
    if bgr is not None:
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        res = inspector.predict(rgb)
        print(f"{label_name} -> is_normal={res.is_normal}, score={res.anomaly_score}, threshold={res.threshold}, time={res.processing_time_ms}ms")

In [ ]:
# 6. Tải trọng số đã huấn luyện về Laptop cá nhân (khi chạy trên Google Colab)
try:
    from google.colab import files
    files.download("weights/patchcore_best.ckpt")
    files.download("weights/metadata.json")
except ImportError:
    print("Đang chạy ở môi trường Local: File đã được lưu sẵn tại weights/patchcore_best.ckpt và weights/metadata.json")